# Gun Detection Training (YOLOv8 + MLflow)

This notebook handles the end-to-end process for Gun Detection:
1. **Data Preparation**: Merges Real and Synthetic datasets (V2/V3), splits them, and formats for YOLO.
2. **Training**: Fine-tunes a YOLOv8 model with MLflow tracking.
3. **Validation**: Evaluates the model on the test split.

In [ ]:
# --- 1. DATA PREPARATION ---
import sys
import os
from src.prepare_data import main as prepare_data

# Ensure src/ is in python path
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)



print("Running Data Preparation...")
# This will:
# 1. Gather files from Real, SynV2, and SynV3 datasets
# 2. Split them into Train/Val/Test (70/20/10)
# 3. Create 'data/images' and 'data/labels' structure
# 4. Generate 'data.yaml'
prepare_data()

Running Data Preparation...
Project Root: D:\_pribadi\gun_detection
Found 652 pairs in Real Dataset
Found 0 pairs in Syn V2 Dataset
Found 893 pairs in Syn V3 Dataset


Processing test: 100%|██████████| 156/156 [00:01<00:00, 129.42it/s]

Created data_real.yaml
Created data_syn_v2.yaml
Created data_syn_v3.yaml
Created data_real_syn_v2.yaml
Created data_real_syn_v3.yaml
Created data_combined.yaml
Created data.yaml
Data preparation complete.


In [ ]:
# --- 2. MODEL TRAINING ---
import mlflow
import torch
from ultralytics import YOLO, settings

# Configuration
mlruns_dir = os.path.join(project_root, "mlruns")
tracking_uri = f"file:///{mlruns_dir}"
model_name = "yolo26s"  # Or 'yolov8n', 'yolov8s', etc.

# MLflow Setup
mlflow.set_tracking_uri(tracking_uri)
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Gun_Detection_Experiment"
settings.update({"mlflow": True})

print(f"Experiment: {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print(f"Tracking URI: {tracking_uri}")

# Initialize Model
model = YOLO(model_name)

# Data Config Path (Generated by prepare_data)
data_config = os.path.join(project_root, "data", "data.yaml")

print(f"Training with config: {data_config}")

# Train
results = model.train(
    data=data_config,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project=os.path.join(project_root, "runs/detect"),
    name="gun_detection_run",
    exist_ok=True,
    verbose=True
)


In [ ]:
# --- 3. VALIDATION ---
print("Validating on Test set...")
metrics = model.val(split='test')
print(f"mAP50-95: {metrics.box.map}")